In [ ]:
import os
import gc
#from dotenv import load_dotenv
#from sqlalchemy.exc import ProgrammingError
from sqlalchemy import create_engine, text, inspect
import pandas as pd
#import cudf

In [ ]:
#create a class for interacting with the postgresql db

class pgsql:
    def __init__(self):
        USER = os.getenv("DB_USER")
        PASSWORD = os.getenv("DB_PASSWORD")
        HOST = os.getenv("DB_HOST")
        PORT = os.getenv("DB_PORT")
        DB_NAME = os.getenv("DB_NAME")
        ADMIN_URL = f"postgresql+psycopg://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
        
        self.engine = create_engine(ADMIN_URL, isolation_level="AUTOCOMMIT")#, echo=True)
        self.inspector = inspect(self.engine)
        self.table_name = ""
        self.pdf = None
    #    self.cudf = None

    def refresh_inspector(self):
        self.inspector = inspect(self.engine)

    def set_table_name(self, table_name = ""):
        self.table_name = table_name

    def get_tables(self):
        return self.inspector.get_table_names()

    def get_sql_description(self, table_name = "", schema = "public"):
        if table_name == "":
            table_name = self.table_name
        columns = self.inspector.get_columns(table_name, schema=schema)
        primary_key_columns = self.inspector.get_pk_constraint(table_name, schema=schema).get("constrained_columns", [])
        return[
            {
                "name": column["name"],
                "type": column["type"],
                "nullable": column["nullable"],
                "default": column["default"],
                "primary_key": column["name"] in primary_key_columns
            } for column in columns
        ]
    
    def set_pdf(self, pdf = pd.DataFrame):
        self.pdf = pdf

    #def set_cudf(self, cudf = None):
    #    self.cudf = cudf

    def get_pdf(self):
        return self.pdf

    #def get_cudf(self):
    #    return self.cudf

    def clear_pdf(self):
        self.pdf = None

    #def clear_cudf(self):
    #    self.cudf = None

    def clear_dataframes(self):
        self.clear_pdf()
        self.clear_cudf()

    def check_pdf_compatability(self):
        #Check if columns names and types of pandas dataframe and sql table are compatible
        if self.pdf is None:
            return False
        sql_description = self.get_sql_description()
        sql_columns = {col["name"]: col["type"] for col in sql_description}
        for col in self.pdf.columns:
            if col not in sql_columns:
                return False
            # You can add more type checking here if needed
            if not pd.api.types.is_dtype_equal(self.pdf[col].dtype, sql_columns[col].python_type):
                return False
        return True

    #def check_cudf_compatability(self):
    #    #Check if columns names and types of cudf dataframe and sql table are compatible
    #    if self.cudf is None:
    #        return False
    #    sql_description = self.get_sql_description()
    #    sql_columns = {col["name"]: col["type"] for col in sql_description}
    #    for col in self.cudf.columns:
    #        if col not in sql_columns:
    #            return False
    #        # You can add more type checking here if needed
    #        if not pd.api.types.is_dtype_equal(self.cudf[col].dtype, sql_columns[col].python_type):
    #            return False
    #    return True

    def write_pdf_to_sql(self):
        if not self.check_pdf_compatability():
            raise ValueError("PDF is not compatible with SQL table")
        self.pdf.to_sql(self.table_name, self.engine, if_exists="replace", index=False)

    #def write_cudf_to_sql(self):
    #    if not self.check_cudf_compatability():
    #        raise ValueError("CUDF is not compatible with SQL table")
    #    self.cudf.to_pandas().to_sql(self.table_name, self.engine, if_exists="replace", index=False)

    def get_table_as_pdf(self):
        self.pdf = pd.read_sql_table(self.table_name, self.engine)
        return self.pdf

    #def get_table_as_cudf(self):
    #    self.cudf = cudf.read_sql_table(self.table_name, self.engine)
    #    return self.cudf


    #next we need a function that can return a subset of the sql table using a SQL query as a string
    def get_table_subset_as_pdf(self, query):
        self.pdf = pd.read_sql_query(query, self.engine)
        return self.pdf

    #def get_table_subset_as_cudf(self, query):
    #    self.cudf = cudf.read_sql_query(query, self.engine)
    #    return self.cudf

    def get_table_subset_as_pdf_with_where_clauses(self, where_clauses={}):
        query = "SELECT * FROM " + self.table_name + " WHERE " + " AND ".join([f"{k} = {v}" for k, v in where_clauses.items()])
        self.pdf = pd.read_sql_query(query, self.engine)
        return self.pdf

    #def get_table_subset_as_cudf_with_where_clauses(self, where_clauses={}):
    #    query = "SELECT * FROM " + self.table_name + " WHERE " + " AND ".join([f"{k} = {v}" for k, v in where_clauses.items()])
    #    self.cudf = cudf.read_sql_query(query, self.engine)
    #    return self.cudf
#where_clauses = {"year": 2024, "month": 1, "day_of_month": 1}
#query = "SELECT * FROM dim_date WHERE " + " AND ".join([f"{k} = {v}" for k, v in where_clauses.items()]) + " LIMIT 10"

In [23]:
test = pgsql()

In [24]:
tables = test.get_tables()
tables

['projects', 'alpaca_assets', 'fred_indicators', 'dim_date']

In [25]:
test.set_table_name("dim_date")

In [26]:
description = test.get_sql_description()
description

[{'name': 'id',
  'type': BIGINT(),
  'nullable': False,
  'default': None,
  'primary_key': True},
 {'name': 'date',
  'type': TIMESTAMP(timezone=True),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'year',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'quarter',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'month',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'day_of_month',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'hour',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False},
 {'name': 'minute',
  'type': INTEGER(),
  'nullable': False,
  'default': None,
  'primary_key': False}]

In [27]:
limit_10 = test.get_table_subset_as_pdf("SELECT * FROM dim_date LIMIT 10")
limit_10

,id,date,year,quarter,month,day_of_month,hour,minute
0,0,1990-01-01 00:00:00+00:00,1990,1,1,1,0,0
1,1,1990-01-01 00:01:00+00:00,1990,1,1,1,0,1
2,2,1990-01-01 00:02:00+00:00,1990,1,1,1,0,2
3,3,1990-01-01 00:03:00+00:00,1990,1,1,1,0,3
4,4,1990-01-01 00:04:00+00:00,1990,1,1,1,0,4
5,5,1990-01-01 00:05:00+00:00,1990,1,1,1,0,5
6,6,1990-01-01 00:06:00+00:00,1990,1,1,1,0,6
7,7,1990-01-01 00:07:00+00:00,1990,1,1,1,0,7
8,8,1990-01-01 00:08:00+00:00,1990,1,1,1,0,8
9,9,1990-01-01 00:09:00+00:00,1990,1,1,1,0,9


In [28]:
test.get_table_subset_as_pdf("SELECT * FROM dim_date WHERE date = '2024-01-01' LIMIT 10")

,id,date,year,quarter,month,day_of_month,hour,minute
0,17882400,2024-01-01 08:00:00+00:00,2024,1,1,1,8,0


In [29]:
test.get_table_subset_as_pdf("SELECT * FROM dim_date WHERE year = 2024 and month = 1  and day_of_month = 1 LIMIT 10")

,id,date,year,quarter,month,day_of_month,hour,minute
0,17881920,2024-01-01 00:00:00+00:00,2024,1,1,1,0,0
1,17881921,2024-01-01 00:01:00+00:00,2024,1,1,1,0,1
2,17881922,2024-01-01 00:02:00+00:00,2024,1,1,1,0,2
3,17881923,2024-01-01 00:03:00+00:00,2024,1,1,1,0,3
4,17881924,2024-01-01 00:04:00+00:00,2024,1,1,1,0,4
5,17881925,2024-01-01 00:05:00+00:00,2024,1,1,1,0,5
6,17881926,2024-01-01 00:06:00+00:00,2024,1,1,1,0,6
7,17881927,2024-01-01 00:07:00+00:00,2024,1,1,1,0,7
8,17881928,2024-01-01 00:08:00+00:00,2024,1,1,1,0,8
9,17881929,2024-01-01 00:09:00+00:00,2024,1,1,1,0,9


In [30]:
where_clauses = {"year": 2024, "month": 1, "day_of_month": 1, "hour": 0}
query = "SELECT * FROM dim_date WHERE " + " AND ".join([f"{k} = {v}" for k, v in where_clauses.items()]) + " LIMIT 10"
test.get_table_subset_as_pdf(query)

,id,date,year,quarter,month,day_of_month,hour,minute
0,17881920,2024-01-01 00:00:00+00:00,2024,1,1,1,0,0
1,17881921,2024-01-01 00:01:00+00:00,2024,1,1,1,0,1
2,17881922,2024-01-01 00:02:00+00:00,2024,1,1,1,0,2
3,17881923,2024-01-01 00:03:00+00:00,2024,1,1,1,0,3
4,17881924,2024-01-01 00:04:00+00:00,2024,1,1,1,0,4
5,17881925,2024-01-01 00:05:00+00:00,2024,1,1,1,0,5
6,17881926,2024-01-01 00:06:00+00:00,2024,1,1,1,0,6
7,17881927,2024-01-01 00:07:00+00:00,2024,1,1,1,0,7
8,17881928,2024-01-01 00:08:00+00:00,2024,1,1,1,0,8
9,17881929,2024-01-01 00:09:00+00:00,2024,1,1,1,0,9


In [31]:
test.get_table_subset_as_pdf_with_where_clauses(where_clauses)

,id,date,year,quarter,month,day_of_month,hour,minute
0,17881920,2024-01-01 00:00:00+00:00,2024,1,1,1,0,0
1,17881921,2024-01-01 00:01:00+00:00,2024,1,1,1,0,1
2,17881922,2024-01-01 00:02:00+00:00,2024,1,1,1,0,2
3,17881923,2024-01-01 00:03:00+00:00,2024,1,1,1,0,3
4,17881924,2024-01-01 00:04:00+00:00,2024,1,1,1,0,4
5,17881925,2024-01-01 00:05:00+00:00,2024,1,1,1,0,5
6,17881926,2024-01-01 00:06:00+00:00,2024,1,1,1,0,6
7,17881927,2024-01-01 00:07:00+00:00,2024,1,1,1,0,7
8,17881928,2024-01-01 00:08:00+00:00,2024,1,1,1,0,8
9,17881929,2024-01-01 00:09:00+00:00,2024,1,1,1,0,9


In [33]:
test.get_pdf().head()

,id,date,year,quarter,month,day_of_month,hour,minute
0,17881920,2024-01-01 00:00:00+00:00,2024,1,1,1,0,0
1,17881921,2024-01-01 00:01:00+00:00,2024,1,1,1,0,1
2,17881922,2024-01-01 00:02:00+00:00,2024,1,1,1,0,2
3,17881923,2024-01-01 00:03:00+00:00,2024,1,1,1,0,3
4,17881924,2024-01-01 00:04:00+00:00,2024,1,1,1,0,4
